# Implement the basics of LLMs + tools and Agent Loop 

In [1]:
from openai import OpenAI

import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Just an LLM call

<img src="2026-04-06-13-32-29.png" width=40%>

In [ ]:
def level_1_llm_call(user_input: str) -> str:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=user_input,
        temperature=0.3
    )
    
    return response.output_text

level_1_llm_call("What is the capital of France?")

'The capital of France is **Paris**.'

# Level 2 LLMs + Tools

In [7]:
from pydantic import BaseModel

class WeatherData(BaseModel):
    city: str
    temperature: int
    description: str

def get_weather(user_input: str) -> str:
    """Tool gets the weather for a given city"""
    weather_data_mock = {
        "city": "Paris",
        "temperature": 20,
        "description": "sunny"
    }
    return weather_data_mock

tool_schema = {
        "type": "function",
        "name": "get_weather",
        "description": "Get today's weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "A city name like Paris or London",
                },
            },
            "required": ["city"],
        },
    }

def level_2_llm_with_tool(user_input: str) -> str:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=user_input,
        temperature=0.3,
        tools=[tool_schema]
    )
    return response

response_output = level_2_llm_with_tool("What is the weather in Paris?")
response_output

Response(id='resp_07d9c26295afd58e0069d3a943f210819483d4927405b996e4', created_at=1775479107.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"city":"Paris"}', call_id='call_jxKdAnMtvbw2DSVEQKGPxKVf', name='get_weather', type='function_call', id='fc_07d9c26295afd58e0069d3a944c8788194a5e56afd702e214b', namespace=None, status='completed')], parallel_tool_calls=True, temperature=0.3, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'A city name like Paris or London'}}, 'required': ['city'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description="Get today's weather for a given city.")], top_p=0.98, background=False, completed_at=1775479108.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prom

<img src="2026-04-06-13-39-36.png"> width=50%

In [25]:
import json

def extract_tool_call_from_response(response):
    """
    Extract and display the tool call from a language model response.
    """
    if not hasattr(response, "output") or not response.output:
        print("No tool call found in the response.")
        return
    tool_call = response.output[0]
    print(f"Tool name: {tool_call.name}")
    print(f"Arguments: {tool_call.arguments}")

# Example usage; this requires the response object from previous cells:
# extract_tool_call_from_response(response_output)

def execute_tool(tool_call) -> str:
    """
    Execute a tool call and return the result.
    """
    # Access `arguments` as an attribute, not a dict key.
    tool_args = json.loads(tool_call.arguments)
    # Some tool-calling APIs may give .name as attribute, not a field in arguments
    # Check both common cases:
    tool_name = getattr(tool_call, "name", None) or tool_args.get("name")
    if tool_name == "get_weather":
        print(tool_args)
        return get_weather(tool_args)

# we need to execute the tool
user_input = "What is the weather in Paris?"
response_output = level_2_llm_with_tool(user_input)
tool_call = response_output.output[0]
tool_output = execute_tool(tool_call)
final_answer_from_llm = f"""
the user send this: {user_input} and the tool returned this: {tool_output},
write the final answer to the user:
"""
llm_call_final_answer = level_2_llm_with_tool(final_answer_from_llm)
llm_call_final_answer.output_text

{'city': 'Paris'}


'The weather in Paris is sunny and 20°C.'

## CLAUDIO: I followed, but had the output of the tool been insuficient, what thinking does the LLM do to conclude "I need to take additional steps because this is not properly answered".

# Agent Loop

<img src="2026-04-06-13-51-27.png" width=70%>

In [31]:
from openai import OpenAI
import json

client = OpenAI()

# The same tool, defined manually
tools = [{
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
}]

def execute_tool(name, args):
    if name == "get_temperature":
        temps = {"lisbon": "22°C", "tokyo": "18°C"}
        return temps.get(args["city"].lower(), "Unknown")

# The agent loop — written explicitly
messages = [
    {"role": "system", "content": "You help with weather. Be concise."},
    {"role": "user", "content": "What's the temperature in Lisbon?"}
]

turn = 0
while turn < 10:  # max_turns safety
    turn += 1
    print(f"--- Loop iteration {turn} ---")
    
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages,
        tools=tools,
    )
    
    message = response.choices[0].message
    messages.append(message)
    
    # EXIT CONDITION: no tool calls → the model decided to respond
    if not message.tool_calls:
        print(f"Model responded: {message.content}")
        break
    
    # CONTINUE: execute tool calls, feed results back
    for tc in message.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"Model called: {tc.function.name}({args})")
        result = execute_tool(tc.function.name, args)
        print(f"Tool returned: {result}")
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result
        })

--- Loop iteration 1 ---
Model called: get_temperature({'city': 'Lisbon'})
Tool returned: 22°C
--- Loop iteration 2 ---
Model responded: The current temperature in Lisbon is 22°C. Want the forecast or any other details?
